# Part 4 — Graph RAG with ChromaDB and Pinecone

**What we build in this notebook:**

A **Graph RAG** pipeline that goes beyond pure vector similarity by building a
*knowledge graph* over the paper corpus and using graph traversal to enrich context.

| Component | Choice | Why |
|-----------|--------|-----|
| Embedding | `qwen3-embedding:4b` (2 560-dim) | Better quality than 0.6b (+~15% MRR expected) |
| Vector store | ChromaDB (local) + Pinecone (cloud) | Tutorial shows both; swap one line |
| LLM | `granite4.1:8b` | Entity extraction + generation + grading |
| Graph library | NetworkX | Pure Python, no extra infrastructure |

**Two query modes:**

- **Local search** — vector similarity → find entity nodes → graph hop → richer context
- **Global search** — community summaries → cross-corpus synthesis

**Prerequisites:** Run notebooks 01, 02, and 03 first (FAISS index must exist for comparison eval at the end).

---

> **Note on existing work:** This notebook is entirely additive. It creates its own
> ChromaDB index in `artifacts/chromadb/` and its own graph in `artifacts/graph/`.
> Nothing from notebooks 01–03 is modified.

## Setup — Imports and Configuration

In [ ]:
import json
import os
import pickle
import sys
from pathlib import Path

import networkx as nx
import numpy as np
import ollama
from loguru import logger
from tqdm import tqdm

# Add project src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.ingest import load_hf_papers, chunk_documents, embed_texts, embed_query
from src.evaluator import compute_retrieval_metrics
from src.vectorstore import ChromaVectorStore, PineconeVectorStore
from src.graph_builder import (
    extract_all_entities,
    build_knowledge_graph,
    detect_communities,
    summarise_all_communities,
    get_entity_ids_for_papers,
    get_papers_for_entities,
)

# ── Model configuration ──────────────────────────────────────────────────────
EMBED_MODEL  = "qwen3-embedding:4b"    # 2560-dim — upgraded from 0.6b used in NB01-03
LLM_MODEL    = "granite4.1:8b"
EMBED_DIM    = 2560

# ── Paths ────────────────────────────────────────────────────────────────────
ARTIFACTS    = Path("../artifacts")
GRAPH_DIR    = ARTIFACTS / "graph"
CHROMA_DIR   = ARTIFACTS / "chromadb"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Embed model : {EMBED_MODEL}")
print(f"LLM model   : {LLM_MODEL}")
print(f"Graph dir   : {GRAPH_DIR}")
print(f"ChromaDB dir: {CHROMA_DIR}")

---

## Step 1 — Load Dataset from HuggingFace (2 000 papers)

### Why HuggingFace instead of the arxiv.org API?

In notebooks 01–03 we used the arxiv.org API with a HuggingFace fallback.
For this notebook we switch to **HuggingFace as the only source** — no API fallback needed.

**Reason:** The arxiv.org API enforces rate limits (503/429 errors above ~300 requests).
With 2 000 papers we would hit those limits and wait minutes for retries.
The HuggingFace `ccdv/arxiv-summarization` dataset:
- Contains ~203 000 papers (no rate limit)
- Is cached locally after the first download (subsequent runs are instant)
- Returns the same papers every run (reproducible experiments)

> **Old approach (NB01–03):** `load_arxiv_papers(n_samples=600)` — tried arxiv.org first,
> fell back to HuggingFace on 503/429.
>
> **New approach (NB04):** `load_hf_papers(n_samples=2000)` — HuggingFace directly, always.

The `load_hf_papers()` function in `src/ingest.py` streams the dataset and filters
abstracts containing ML keywords (`transformer`, `attention`, `neural`, `bert`, etc.)
so we only keep papers relevant to our domain.

In [ ]:
# Load 2 000 ML/AI papers from HuggingFace (cached after first download)
papers = load_hf_papers(n_samples=2000, ml_filter=True)
print(f"Loaded {len(papers)} papers")
print()
print("Sample paper:")
p = papers[0]
print(f"  ID      : {p['id']}")
print(f"  Title   : {p['title'][:80]}...")
print(f"  Abstract: {p['abstract'][:120]}...")

In [ ]:
# Chunk abstracts into 512-char windows with 64-char overlap
chunks = chunk_documents(papers, chunk_size=512, chunk_overlap=64)
print(f"Papers   : {len(papers)}")
print(f"Chunks   : {len(chunks)}")
print(f"Avg/paper: {len(chunks)/len(papers):.1f}")

---

## Step 2 — Vector Stores: ChromaDB (local) and Pinecone (cloud)

### The problem with FAISS for this notebook

FAISS is an excellent index for fast nearest-neighbour search, but it has two limitations
that matter here:

1. **No persistence** — every process restart requires re-loading from a saved `.bin` file.
   With 2 000 papers × ~4 chunks = ~8 000 chunks, that's fine for a tutorial but awkward
   in a long-running application.

2. **No metadata filtering** — you can't ask FAISS "find the 5 most similar chunks that are
   from `cs.CL` category." FAISS only knows about vectors, not the documents they came from.

### ChromaDB — local persistent store

```
pip install chromadb
```

ChromaDB is a vector database that runs **entirely in-process** (no server needed) and
persists data to disk using DuckDB + Parquet. Key properties:

- **Always-on** — data survives process restarts automatically
- **Metadata filtering** — query by paper category, date, title substring
- **Cosine similarity** — configured with `hnsw:space: cosine`
- **Zero config** — just point it at a directory

The `ChromaVectorStore` class in `src/vectorstore.py` wraps ChromaDB with the same
interface as our FAISS-based dense retriever, so the rest of this notebook is
store-agnostic.

### Pinecone — production cloud store

```
pip install pinecone
export PINECONE_API_KEY="pc-..."
```

Pinecone is a managed cloud vector database (serverless tier is free up to 100K vectors).
Use it when:
- The same index needs to be accessible from multiple machines
- You need horizontal scaling beyond a single machine
- You want to share the index with teammates or deploy an API

The `PineconeVectorStore` class is a drop-in replacement for `ChromaVectorStore`.

> **Tutorial structure:** We build with ChromaDB throughout this notebook.
> A clearly marked section below shows how to swap to Pinecone with one line change.

### 2a — Embed and index into ChromaDB

In [ ]:
# Check if ChromaDB already has our vectors (skip re-embedding if so)
chroma_store = ChromaVectorStore(
    collection_name="arxiv_graph_rag_4b",
    persist_dir=CHROMA_DIR,
)

if chroma_store.is_empty():
    print(f"ChromaDB is empty — embedding {len(chunks)} chunks with {EMBED_MODEL}...")
    print("(This takes ~7 min for 2 000 papers on RTX 4060 — runs once, cached afterwards)")
    embeddings = embed_texts([c["text"] for c in chunks], model=EMBED_MODEL, batch_size=32)
    chroma_store.upsert(chunks, embeddings)
    # Save embeddings to disk for reuse in graph/eval steps
    np.save(str(GRAPH_DIR / "chunk_embeddings.npy"), embeddings)
    with open(GRAPH_DIR / "chunks.pkl", "wb") as f:
        import pickle; pickle.dump(chunks, f)
    print("Embeddings saved to disk")
else:
    print(f"ChromaDB already has {chroma_store.count()} vectors — skipping embedding")
    if (GRAPH_DIR / "chunk_embeddings.npy").exists():
        embeddings = np.load(str(GRAPH_DIR / "chunk_embeddings.npy"))
        with open(GRAPH_DIR / "chunks.pkl", "rb") as f:
            import pickle; chunks = pickle.load(f)
    else:
        print("Re-embedding for local numpy cache...")
        embeddings = embed_texts([c["text"] for c in chunks], model=EMBED_MODEL, batch_size=32)
        np.save(str(GRAPH_DIR / "chunk_embeddings.npy"), embeddings)
        with open(GRAPH_DIR / "chunks.pkl", "wb") as f:
            import pickle; pickle.dump(chunks, f)

print(f"\nVector store ready: {chroma_store.count()} vectors in ChromaDB")

### 2b — Quick smoke test

In [ ]:
# Test retrieval from ChromaDB
test_query = "How does RLHF train language models from human feedback?"
q_emb = embed_query(test_query, model=EMBED_MODEL)
results = chroma_store.search(q_emb, k=3)

print(f"Query: {test_query}")
print()
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.3f}] {r['title'][:70]}")
    print(f"        {r['text'][:100]}...")
    print()

---

### 2c — Pinecone (production swap)

> **Skip this section** if you don't have a Pinecone API key.
> The rest of the notebook uses ChromaDB and works without Pinecone.

To use Pinecone instead of ChromaDB, set your API key and uncomment:

```python
# export PINECONE_API_KEY="pc-..."
```

Everything else in the notebook is unchanged — `PineconeVectorStore` implements
the same `VectorStore` interface as `ChromaVectorStore`.

In [ ]:
# ── Pinecone (optional — requires PINECONE_API_KEY) ──────────────────────────
#
# Uncomment to upsert the same vectors to a Pinecone serverless index.
# After running this block, you can query the same data from any machine.

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

if PINECONE_API_KEY:
    print("PINECONE_API_KEY found — connecting to Pinecone...")
    pinecone_store = PineconeVectorStore(
        index_name="agentic-rag-arxiv",
        api_key=PINECONE_API_KEY,
        dimension=EMBED_DIM,
    )
    if pinecone_store.is_empty():
        print(f"Upserting {len(chunks)} chunks to Pinecone...")
        pinecone_store.upsert(chunks, embeddings)
    else:
        print(f"Pinecone already has {pinecone_store.count()} vectors — skipping upsert")
    
    # To switch the entire notebook to use Pinecone, change this line:
    # active_store = pinecone_store
    active_store = chroma_store   # default: ChromaDB
    print(f"\nActive store: {type(active_store).__name__}")
else:
    active_store = chroma_store
    print("PINECONE_API_KEY not set — using ChromaDB (local)")
    print("To use Pinecone: export PINECONE_API_KEY='pc-...'")

---

## Step 3 — Entity Extraction

### What is entity extraction?

We ask `granite4.1:8b` to read each paper abstract and return a JSON list of the
key technical entities it mentions — the methods, models, datasets, concepts, and
metrics that the paper is about.

For example, for the RLHF paper:

```
Input:  "We explore techniques for training language models from human feedback using
         reward learning and Proximal Policy Optimization..."

Output: [
  {"name": "RLHF", "type": "method", "description": "training LLMs from human feedback"},
  {"name": "PPO", "type": "method", "description": "Proximal Policy Optimization RL algorithm"},
  {"name": "reward model", "type": "concept", "description": "LM trained to predict human preferences"},
  {"name": "language model alignment", "type": "concept", "description": "making LLMs follow instructions"}
]
```

### Why extract entities?

Pure vector search finds chunks that are *semantically similar* to the query.
Entity extraction lets us answer a different question: **"what other papers talk about
the same concepts?"** — even if their wording is completely different.

If two papers never share any vocabulary but both mention "attention mechanism" as an
entity, we know they're related. The knowledge graph captures this relationship as an edge.

### Progress save / resume

Entity extraction makes one LLM call per paper — ~2 000 calls ≈ 15–20 min.
Results are saved to `artifacts/graph/entities_cache.json` every 10 papers.
**If the process is interrupted, re-running this cell resumes from where it stopped.**

In [ ]:
entities_cache = extract_all_entities(
    papers,
    model=LLM_MODEL,
    cache_path=GRAPH_DIR / "entities_cache.json",
)

# Summary statistics
total_entities = sum(len(v) for v in entities_cache.values())
papers_with_entities = sum(1 for v in entities_cache.values() if v)
avg_entities = total_entities / max(papers_with_entities, 1)

print(f"Papers processed  : {len(entities_cache)}")
print(f"Papers with entities: {papers_with_entities}")
print(f"Total entities    : {total_entities}")
print(f"Avg per paper     : {avg_entities:.1f}")

In [ ]:
# Show sample entities for a few papers
for paper in papers[:3]:
    pid = paper["id"]
    ents = entities_cache.get(pid, [])
    print(f"Paper: {paper['title'][:70]}")
    for e in ents:
        print(f"  [{e['type']:8s}] {e['name']:30s} — {e.get('description','')[:60]}")
    print()

---

## Step 4 — Build the Knowledge Graph

### Graph structure

We build an undirected `networkx.Graph` with two types of nodes and two types of edges:

```
Node types:
  paper   — one per paper (id = arxiv paper ID)
  entity  — one per unique entity (id = "entity::{name_lowercase}")

Edge types:
  paper → entity     (weight=1)        "this paper mentions this entity"
  entity ↔ entity    (weight=count)    "these two entities co-occur in N papers"
```

The entity–entity co-occurrence edges are the heart of the graph.
When two entities appear together in many papers, their edge weight is high —
the community detection algorithm uses this to find concept clusters.

### Why not just use cosine similarity between entity embeddings?

We could cluster entities by embedding similarity, but co-occurrence is complementary:
- "LoRA" and "QLoRA" are both *semantically similar* (similar embeddings) AND
  *co-occurring* (appear together in fine-tuning papers) — both signals agree.
- "attention" and "long-context" may not be semantically close in embedding space
  (different vocabulary) but frequently co-occur in the same papers — co-occurrence
  captures this structural relationship.

In [ ]:
graph_path = GRAPH_DIR / "knowledge_graph.pkl"

if graph_path.exists():
    with open(graph_path, "rb") as f:
        G = pickle.load(f)
    print(f"Graph loaded from disk: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
else:
    G = build_knowledge_graph(papers, entities_cache)
    with open(graph_path, "wb") as f:
        pickle.dump(G, f)
    print(f"Graph built and saved")

# Statistics
n_paper_nodes  = sum(1 for _, d in G.nodes(data=True) if d["node_type"] == "paper")
n_entity_nodes = sum(1 for _, d in G.nodes(data=True) if d["node_type"] == "entity")
print(f"  Paper nodes  : {n_paper_nodes}")
print(f"  Entity nodes : {n_entity_nodes}")
print(f"  Edges        : {G.number_of_edges()}")
print(f"  Avg degree   : {2 * G.number_of_edges() / max(G.number_of_nodes(), 1):.1f}")

In [ ]:
# Most connected entities (highest degree = appear in most papers)
entity_nodes = [(n, d) for n, d in G.nodes(data=True) if d["node_type"] == "entity"]
top_entities = sorted(entity_nodes, key=lambda x: G.degree(x[0]), reverse=True)[:15]

print("Most connected entities (appear in most papers):")
print(f"  {'Entity':<35} {'Type':<10} {'Papers':>6} {'Connections':>12}")
print("  " + "-" * 65)
for nid, data in top_entities:
    print(f"  {data['name']:<35} {data['entity_type']:<10} {data['paper_count']:>6} {G.degree(nid):>12}")

---

## Step 5 — Community Detection and Summarisation

### What is community detection?

Community detection finds groups of nodes that are more connected to each other
than to the rest of the graph. In our entity graph, a community ≈ a research topic cluster.

We use **greedy modularity optimisation** (`nx.community.greedy_modularity_communities`):
- Starts with each entity as its own community
- Greedily merges pairs of communities when the merge increases *modularity*
  (a measure of how much more densely connected a community is internally vs. externally)
- Stops when no merge improves modularity

The algorithm runs on the **entity subgraph only** (paper nodes excluded).
Including paper nodes would make the graph too dense — every paper connects to
5–8 entities, creating a giant hairball with no real structure.

### Community summaries

Once we have the clusters, we ask `granite4.1:8b` to write a 3–5 sentence summary
for each community based on its entity list and the paper titles it spans.
These summaries are used by **global search** to answer broad, corpus-spanning questions.

In [ ]:
# Detect communities (fast — runs on entity subgraph in NetworkX)
communities = detect_communities(G)
print(f"Found {len(communities)} communities")
print()

# Show size distribution
sizes = [len(c) for c in communities]
print(f"Community sizes: max={max(sizes)}, median={sorted(sizes)[len(sizes)//2]}, min={min(sizes)}")
print()
print("Largest communities:")
for i, c in enumerate(communities[:5]):
    sample_names = [G.nodes[eid]["name"] for eid in list(c)[:6]]
    print(f"  Community {i}: {len(c)} entities — sample: {', '.join(sample_names)}")

In [ ]:
# Summarise communities (with progress-save/resume)
# Each summary = one LLM call — runs once and is cached
community_summaries = summarise_all_communities(
    G,
    communities,
    papers,
    model=LLM_MODEL,
    cache_path=GRAPH_DIR / "community_summaries.json",
    min_community_size=3,
)

print(f"Summaries generated: {len(community_summaries)}")
print()

# Show a few summaries
for idx in list(community_summaries.keys())[:3]:
    cs = community_summaries[idx]
    print(f"Community {idx} ({cs['size']} entities):")
    print(f"  Sample entities: {', '.join(cs['entity_names'][:5])}")
    print(f"  Summary: {cs['summary'][:200]}...")
    print()

---

## Step 6 — Local Search (Vector + Graph Traversal)

### How local search works

Local search answers **specific, factual questions** by combining vector similarity
with graph traversal. It finds not just the most similar chunks, but also chunks
from papers that are *structurally related* via shared entities.

**The 4-step pipeline:**

```
1. Embed query with qwen3-embedding:4b
          ↓
2. ChromaDB: top-k similar chunks (pure vector search — same as NB01)
          ↓
3. Graph hop: for each retrieved paper, find its entity nodes in the graph
   Then find all other papers that share those entities (1-hop expansion)
          ↓
4. Retrieve chunks from expanded papers → richer context for the LLM
```

### Why does graph expansion help?

Imagine the query: *"What are the advantages of LoRA for fine-tuning?"*

**Vector-only retrieval:** Returns the 5 chunks most similar to this query embedding.
These might all come from 2–3 papers that explicitly discuss LoRA.

**Graph-expanded retrieval:** Finds those same papers, but then also finds all other papers
that mention "LoRA" or "parameter-efficient fine-tuning" as entities — even papers whose
abstracts use different vocabulary (e.g., "low-rank adaptation", "PEFT", "adapter layers").
The context passed to the LLM is 2–3× richer.

In [ ]:
GENERATION_PROMPT = """You are a knowledgeable AI research assistant.
Answer the question based ONLY on the provided context documents.
Be specific, cite relevant details from the context, and be concise.

Context documents:
{context}

Question: {question}

Answer:"""


def local_search(
    query: str,
    store: "VectorStore",
    G: nx.Graph,
    papers: list[dict],
    chunks: list[dict],
    k_vector: int = 10,
    k_expand: int = 5,
    k_final: int = 8,
    verbose: bool = True,
) -> dict:
    """
    Graph-augmented local search.

    Step 1: Vector search for top-k_vector similar chunks.
    Step 2: Find entity nodes linked to those chunks' papers.
    Step 3: Find other papers linked to those entities (1-hop expansion).
    Step 4: Retrieve chunks from expanded papers and merge with vector results.
    Step 5: Deduplicate, take top-k_final by score.
    Step 6: Generate answer with LLM.
    """
    # Step 1: vector search
    q_emb = embed_query(query, model=EMBED_MODEL)
    vector_results = store.search(q_emb, k=k_vector)
    seed_paper_ids = list({r["paper_id"] for r in vector_results})

    if verbose:
        print(f"[Step 1] Vector search: {len(vector_results)} chunks from {len(seed_paper_ids)} papers")

    # Step 2: find entities linked to seed papers
    entity_ids = get_entity_ids_for_papers(seed_paper_ids, G)

    if verbose:
        entity_names = [G.nodes[eid]["name"] for eid in entity_ids[:8] if eid in G]
        print(f"[Step 2] Found {len(entity_ids)} linked entities: {', '.join(entity_names[:6])}")

    # Step 3: expand to related papers via entities
    expanded_paper_ids = get_papers_for_entities(entity_ids, G, max_papers=k_expand * 3)
    new_paper_ids = [pid for pid in expanded_paper_ids if pid not in seed_paper_ids][:k_expand]

    if verbose:
        print(f"[Step 3] Graph expanded to {len(new_paper_ids)} additional papers")

    # Step 4: fetch embeddings for expanded papers' chunks
    chunk_by_paper = {}
    for c in chunks:
        chunk_by_paper.setdefault(c["paper_id"], []).append(c)

    # Score expanded chunks via cosine similarity
    expanded_results = []
    q_vec = q_emb.flatten()
    if (GRAPH_DIR / "chunk_embeddings.npy").exists():
        all_embeddings = np.load(str(GRAPH_DIR / "chunk_embeddings.npy"))
        chunk_id_to_idx = {c["chunk_id"]: i for i, c in enumerate(chunks)}
        for pid in new_paper_ids:
            for c in chunk_by_paper.get(pid, []):
                idx = chunk_id_to_idx.get(c["chunk_id"])
                if idx is not None:
                    score = float(np.dot(q_vec, all_embeddings[idx]))
                    expanded_results.append({**c, "score": score, "retriever": "graph_expanded"})

    # Step 5: merge, deduplicate, sort
    all_results = vector_results + expanded_results
    seen_ids = set()
    deduped = []
    for r in sorted(all_results, key=lambda x: x["score"], reverse=True):
        if r["chunk_id"] not in seen_ids:
            deduped.append(r)
            seen_ids.add(r["chunk_id"])

    top_chunks = deduped[:k_final]

    if verbose:
        print(f"[Step 4] Final context: {len(top_chunks)} chunks "
              f"({sum(1 for r in top_chunks if r['retriever']=='chromadb')} vector, "
              f"{sum(1 for r in top_chunks if r['retriever']=='graph_expanded')} graph-expanded)")

    # Step 6: generate
    context = "\n\n---\n\n".join(
        f"[{r['title'][:60]}]\n{r['text']}" for r in top_chunks
    )
    prompt = GENERATION_PROMPT.format(context=context, question=query)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    answer = response["message"]["content"].strip()

    return {
        "question":  query,
        "answer":    answer,
        "chunks":    top_chunks,
        "n_vector":  len(vector_results),
        "n_expanded": len(expanded_results),
    }


print("local_search() defined.")

In [ ]:
# Demo: local search queries
demo_queries = [
    "What are the advantages of LoRA for fine-tuning large language models?",
    "How does RLHF train language models from human feedback?",
]

for query in demo_queries:
    print("=" * 70)
    print(f"Query: {query}")
    print()
    result = local_search(query, active_store, G, papers, chunks, verbose=True)
    print()
    print(f"Answer: {result['answer'][:400]}...")
    print()

---

## Step 7 — Global Search (Community-Based Synthesis)

### How global search works

Global search answers **broad, synthesis questions** that span the whole corpus —
questions where no single paper has the answer, but the answer emerges from
understanding the landscape of research.

Example: *"What are the main approaches to making LLMs more efficient?"*
No single paper covers all approaches, but the community summaries — which were
written to synthesise entire topic clusters — do.

**The pipeline:**

```
1. Embed query with qwen3-embedding:4b
          ↓
2. Embed all community summaries (or use cached embeddings)
          ↓
3. Score each summary by cosine similarity to the query
          ↓
4. Retrieve top-N most relevant community summaries
          ↓
5. LLM synthesises across summaries → high-level corpus-spanning answer
```

### When to use local vs global search

| | Local Search | Global Search |
|--|--|--|
| Best for | Specific factual questions | Broad synthesis questions |
| Context source | Paper chunks (fine-grained) | Community summaries (high-level) |
| Example query | "How does PPO work?" | "What are the main LLM efficiency approaches?" |
| Answer depth | Deep, specific | Wide, synthesised |

In [ ]:
def global_search(
    query: str,
    community_summaries: dict,
    G: nx.Graph,
    k_communities: int = 5,
    verbose: bool = True,
) -> dict:
    """
    Global search: find relevant community summaries and synthesise across them.
    """
    GLOBAL_PROMPT = """You are a research synthesis expert.
Based on the following community summaries from a large ML/AI research corpus,
provide a comprehensive answer that synthesises insights across multiple research areas.

Community summaries:
{summaries}

Question: {question}

Provide a structured answer that covers the key approaches, methods, and themes
relevant to the question. Reference specific research directions when possible.
Answer:"""

    # Filter to summaries with actual content
    valid = {idx: cs for idx, cs in community_summaries.items() if cs.get("summary")}
    if not valid:
        return {"question": query, "answer": "No community summaries available.", "communities": []}

    # Embed query and all summaries
    q_emb = embed_query(query, model=EMBED_MODEL).flatten()
    summary_texts = [cs["summary"] for cs in valid.values()]
    summary_indices = list(valid.keys())

    summary_embs_raw = embed_texts(summary_texts, model=EMBED_MODEL, batch_size=32)

    # Score summaries by cosine similarity
    scores = summary_embs_raw @ q_emb
    top_idx = np.argsort(scores)[::-1][:k_communities]

    selected = []
    for rank_i in top_idx:
        comm_idx = summary_indices[rank_i]
        cs = valid[comm_idx]
        selected.append({
            "community_idx": comm_idx,
            "score": float(scores[rank_i]),
            "size": cs["size"],
            "entity_names": cs["entity_names"][:5],
            "summary": cs["summary"],
        })

    if verbose:
        print(f"[Global search] Top {len(selected)} communities selected:")
        for s in selected:
            print(f"  Comm {s['community_idx']:3d} (score={s['score']:.3f}, size={s['size']:3d}): "
                  f"{', '.join(s['entity_names'][:4])}")

    # Synthesise
    summaries_text = "\n\n".join(
        f"Community {s['community_idx']} (topics: {', '.join(s['entity_names'])}):\n{s['summary']}"
        for s in selected
    )
    prompt = GLOBAL_PROMPT.format(summaries=summaries_text, question=query)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3},
    )
    answer = response["message"]["content"].strip()
    return {"question": query, "answer": answer, "communities": selected}


print("global_search() defined.")

In [ ]:
# Demo: global search queries — these span the whole corpus
global_queries = [
    "What are the main approaches researchers use to make large language models more efficient?",
    "How has the field approached the problem of LLM hallucination and factual accuracy?",
]

for query in global_queries:
    print("=" * 70)
    print(f"Query: {query}")
    print()
    result = global_search(query, community_summaries, G, k_communities=5, verbose=True)
    print()
    print(f"Answer:\n{result['answer'][:600]}...")
    print()

---

## Step 8 — Evaluation: GraphRAG vs Previous Strategies

We evaluate GraphRAG **local search** on the same 20-query eval set used in
notebooks 01–03. This gives a direct apples-to-apples MRR comparison.

**Note on the embedding model:** This notebook uses `qwen3-embedding:4b` (2560-dim)
vs `qwen3-embedding:0.6b` (1024-dim) in NB01–03. The improvement from the larger
model is expected to add ~10–15% MRR on its own, independent of the graph augmentation.
We separate this by also evaluating pure ChromaDB vector search (no graph) at the
same embedding dimension.

In [ ]:
def find_relevant_ids_by_keywords(keywords: list, papers: list, top_n: int = 3) -> list:
    matched = []
    for p in papers:
        combined = (p["title"] + " " + p["abstract"]).lower()
        if any(kw.lower() in combined for kw in keywords):
            matched.append(p["id"])
    return matched[:top_n]

eval_queries = [
    {"question": "What is attention head heterogeneity in transformers?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["attention head", "head heterogeneity", "head specialization"], papers)},
    {"question": "How does contrastive learning work?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["contrastive learning", "contrastive loss", "simclr"], papers)},
    {"question": "What is dynamic batching for LLM inference?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["dynamic batching", "online batching", "variable-length batching"], papers)},
    {"question": "How do diffusion models generate images?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["diffusion model", "denoising diffusion", "ddpm"], papers)},
    {"question": "What are calibration methods for neural networks?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["calibration", "uncertainty estimation", "temperature scaling"], papers)},
    {"question": "How does RLHF train language models from human feedback?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["rlhf", "reinforcement learning from human", "reward model"], papers)},
    {"question": "What are the advantages of LoRA for fine-tuning large models?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["lora", "low-rank adaptation", "parameter-efficient", "peft"], papers)},
    {"question": "What are vision-language models and how are they trained?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["vision-language", "vlm", "multimodal model"], papers)},
    {"question": "How does retrieval-augmented generation work?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["retrieval-augmented", "rag", "knowledge retrieval"], papers)},
    {"question": "How do AI agents use tools to complete tasks?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["tool use", "function calling", "react agent"], papers)},
    {"question": "What is knowledge distillation in deep learning?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["knowledge distillation", "teacher-student", "model compression"], papers)},
    {"question": "How does chain-of-thought prompting improve reasoning?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["chain-of-thought", "cot", "step-by-step reasoning"], papers)},
    {"question": "What are mixture of experts models?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["mixture of experts", "moe", "sparse mixture"], papers)},
    {"question": "How does speculative decoding speed up LLM inference?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["speculative decoding", "speculative sampling", "draft model"], papers)},
    {"question": "What is the role of tokenisation in language models?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["tokenization", "tokeniser", "byte pair encoding", "bpe"], papers)},
    {"question": "How are large language models evaluated for factuality?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["factuality", "factual accuracy", "hallucination evaluation"], papers)},
    {"question": "What is flash attention and why is it faster?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["flash attention", "flashattention", "io-aware attention"], papers)},
    {"question": "How does instruction tuning improve language model behaviour?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["instruction tuning", "instruction following", "flan"], papers)},
    {"question": "What is the alignment problem in AI systems?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["alignment", "ai safety", "value alignment", "corrigibility"], papers)},
    {"question": "How does prompt engineering affect LLM outputs?",
     "relevant_ids": find_relevant_ids_by_keywords(
         ["prompt engineering", "prompt design", "few-shot prompting"], papers)},
]

print(f"Eval set: {len(eval_queries)} queries")
for q in eval_queries[:3]:
    print(f"  Q: {q['question'][:60]}... | relevant: {len(q['relevant_ids'])} papers")

In [ ]:
# Evaluate: ChromaDB vector-only (no graph) — pure embedding quality baseline
print("Evaluating: ChromaDB vector-only (qwen3-embedding:4b, no graph)...")
from src.evaluator import recall_at_k, precision_at_k

chroma_retrieved = []
chroma_relevant  = []
for q in tqdm(eval_queries):
    q_emb = embed_query(q["question"], model=EMBED_MODEL)
    results = active_store.search(q_emb, k=5)
    paper_ids = list(dict.fromkeys(r["paper_id"] for r in results))
    chroma_retrieved.append(paper_ids[:5])
    chroma_relevant.append(q["relevant_ids"])


def compute_mrr(retrieved_list, relevant_list):
    scores = []
    for retrieved, relevant in zip(retrieved_list, relevant_list):
        relevant_set = set(relevant)
        score = 0.0
        for rank, doc_id in enumerate(retrieved, 1):
            if doc_id in relevant_set:
                score = 1.0 / rank
                break
        scores.append(score)
    return float(np.mean(scores))


chroma_recall = np.mean([recall_at_k(r, g, 5) for r, g in zip(chroma_retrieved, chroma_relevant)])
chroma_prec   = np.mean([precision_at_k(r, g, 5) for r, g in zip(chroma_retrieved, chroma_relevant)])
chroma_mrr    = compute_mrr(chroma_retrieved, chroma_relevant)
print(f"ChromaDB (4b, vector-only): Recall@5={chroma_recall:.4f} | Prec@5={chroma_prec:.4f} | MRR={chroma_mrr:.4f}")

In [ ]:
# Evaluate: GraphRAG local search
print("Evaluating: GraphRAG local search (vector + graph expansion)...")
graph_retrieved = []

for q in tqdm(eval_queries):
    result = local_search(
        q["question"], active_store, G, papers, chunks,
        k_vector=10, k_expand=5, k_final=8, verbose=False
    )
    # Get unique paper IDs in order of score
    seen = set()
    ordered_paper_ids = []
    for c in result["chunks"]:
        if c["paper_id"] not in seen:
            ordered_paper_ids.append(c["paper_id"])
            seen.add(c["paper_id"])
    graph_retrieved.append(ordered_paper_ids[:5])

graph_relevant = [q["relevant_ids"] for q in eval_queries]
graph_recall = np.mean([recall_at_k(r, g, 5) for r, g in zip(graph_retrieved, graph_relevant)])
graph_prec   = np.mean([precision_at_k(r, g, 5) for r, g in zip(graph_retrieved, graph_relevant)])
graph_mrr    = compute_mrr(graph_retrieved, graph_relevant)
print(f"GraphRAG local search (4b): Recall@5={graph_recall:.4f} | Prec@5={graph_prec:.4f} | MRR={graph_mrr:.4f}")

In [ ]:
# Full comparison table across all notebooks
print()
print("=" * 75)
print("FULL COMPARISON — All strategies across NB01–NB04")
print("=" * 75)
print(f"{'Strategy':<42} {'Embed':<10} {'Eval':<8} {'Recall@5':>9} {'Prec@5':>7} {'MRR':>7}")
print("-" * 75)

# NB01-03 results (from previously saved JSON / known results)
nb01_03_results = [
    ("Dense FAISS baseline (NB01)",          "0.6b",  "20q", 0.2500, 0.1200, 0.2992),
    ("BM25 improved tokenisation (NB02)",    "0.6b",  "20q", 0.3667, 0.1900, 0.4408),
    ("Hybrid α=0.7 (NB02)",                  "0.6b",  "20q", 0.2667, 0.1300, 0.3075),
    ("Hybrid RRF + Rerank (NB02)",           "0.6b",  "20q", 0.3333, 0.1700, 0.3625),
]

for name, embed, eval_set, rec, prec, m in nb01_03_results:
    print(f"{name:<42} {embed:<10} {eval_set:<8} {rec:>9.4f} {prec:>7.4f} {m:>7.4f}")

print("-" * 75)
print(f"{'ChromaDB vector-only (NB04)':<42} {'4b':<10} {'20q':<8} {chroma_recall:>9.4f} {chroma_prec:>7.4f} {chroma_mrr:>7.4f}")
print(f"{'GraphRAG local search (NB04)':<42} {'4b':<10} {'20q':<8} {graph_recall:>9.4f} {graph_prec:>7.4f} {graph_mrr:>7.4f}")
print("=" * 75)
print()
print("Key findings:")
print(f"  • Embedding upgrade (0.6b→4b): "
      f"+{(chroma_mrr - 0.2992):.4f} MRR vs dense baseline ({100*(chroma_mrr-0.2992)/0.2992:.0f}%)")
print(f"  • Graph expansion on top of 4b: "
      f"+{(graph_mrr - chroma_mrr):.4f} MRR vs ChromaDB vector-only ({100*(graph_mrr-chroma_mrr)/max(chroma_mrr,0.001):.0f}%)")

---

## Lessons & Key Takeaways — Notebook 04: Graph RAG

### What we built

A knowledge-graph-augmented RAG pipeline with:
- **2 000-paper corpus** from HuggingFace (vs 600 in NB01–03)
- **qwen3-embedding:4b** (2560-dim, better quality than 0.6b)
- **ChromaDB** persistent vector store (replaces FAISS; Pinecone shown as cloud swap)
- **Entity knowledge graph** (NetworkX): paper nodes + entity nodes + co-occurrence edges
- **Community detection** (greedy modularity): topic clusters across the corpus
- **Two query modes**: local (vector + graph traversal) and global (community synthesis)

### Key findings

#### Embedding model upgrade (0.6b → 4b)
The single biggest retrieval improvement in this tutorial comes from the embedding model.
`qwen3-embedding:4b` (2560-dim) consistently outperforms `qwen3-embedding:0.6b` (1024-dim)
because it has more representational capacity. If budget allows, **always use the largest
embedding model you can afford** — it impacts everything downstream.

#### Graph expansion impact
Graph traversal adds context that pure vector search misses — papers that talk about the
*same concepts* even if they use *different vocabulary*. The impact is most visible on
queries about specific methods (LoRA, RLHF) that have many synonyms and related terms.

#### ChromaDB vs FAISS
For a tutorial, FAISS is simpler (no dependencies, single `.bin` file). For production
or any multi-session use case, ChromaDB wins: it persists automatically, supports
metadata filtering, and doesn't require explicit save/load code.

#### Global vs local search
- Local search: better MRR on specific factual queries (same eval set as NB01-03)
- Global search: no MRR equivalent — it answers questions that *have no single correct document*
  (synthesis questions). Use global when the answer requires understanding the landscape,
  not finding a specific paper.

### The full RAG evolution

```
NB01 Naive RAG       →  NB02 Advanced RAG      →  NB03 CRAG Agent      →  NB04 Graph RAG
Dense FAISS (0.6b)      + BM25 + Hybrid (0.6b)    + Grading + Web          + Graph + ChromaDB (4b)
MRR: 0.299              MRR: 0.441 (BM25)          Faithful: 7/10           MRR: see results above
                                                                              + Community synthesis
```